# 01 · Explore the full NYC 311 dataset
**Team BUTA · DS-GA 1007 · Weather sensitivity of 311 complaints**

**Dataset:** NYC 311 Service Requests, 2020 → present (NYC Open Data `erm2-nwe9`), stored as one parquet file per month in `data/raw/311_full/`.
```bash
python -m src.download --only 311full
```

**Goal of this notebook:** understand the full data before cutting it down.
- How big is it, and how has volume changed over time?
- Which columns are reliable, and which are mostly empty?
- Which complaint types dominate, and **which ones are seasonal**? Seasonal types are candidates for a weather effect.
- What data-quality problems will the cleaning notebook need to handle?

**Memory:** about 23M rows don't fit in pandas on a 16 GB laptop. Every query below therefore streams **one month at a time** and reads only the columns it needs, using `src/transform.py`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src import config, transform as T
from src.plots import save, ACCENT

pd.set_option("display.float_format", "{:,.2f}".format)
files = T.month_files()
print(f"{len(files)} monthly files: {files[0].stem} → {files[-1].stem}")

## 1. Size and volume over time
Row counts come from parquet metadata, so no data is loaded.

In [ ]:
inv = T.inventory(files)
print(f"Total requests: {inv.rows.sum():,} | on disk: {inv.size_mb.sum() / 1000:.2f} GB")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(pd.to_datetime(inv.month), inv.rows / 1000, width=25, color=ACCENT)
ax.set(title="311 requests per month (full dataset)", ylabel="Thousands of requests")
save(fig, "01a_requests_per_month")

## 2. Columns: schema drift and missing values

In [ ]:
presence = T.column_presence(files)
drift = presence[~presence.all(axis=1)].sum(axis=1).rename("months_present")
print(f"{len(presence)} distinct columns; {len(drift)} are missing from at least one month (the API drops all-empty columns)")
drift.to_frame()

In [ ]:
missing = T.missing_counts(files)
missing[["missing", "missing_share"]].style.format({"missing": "{:,.0f}", "missing_share": "{:.1%}"}) \
    .bar(subset="missing_share", color="#d9c2ea")

**Reading it:**
- The core fields (dates, type, agency, location) are nearly complete.
- Many columns are almost always empty (taxi, bridge/highway, vehicle and so on). They only apply to a handful of complaint types.
- We will keep about 17 useful columns.

## 3. What gets reported?

In [ ]:
prof = T.profile(files)
display(prof["year"].sort_index().T)
print(f"Distinct complaint types: {len(prof['complaint_type'])}")

top = prof["complaint_type"].head(20)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index[::-1], top.requests[::-1] / 1e6, color=ACCENT)
ax.set(xlabel="Millions of requests (2020 → now)", title="Top 20 complaint types")
save(fig, "01b_top_complaint_types")

In [ ]:
display(prof["agency"].head(10).T)
display(prof["borough"].T)
prof["status"].T

## 4. How do people report? (channel shift over time)

In [ ]:
def channel_by_year(df):
    return pd.crosstab(df["created_date"].str[:4], df["open_data_channel_type"].fillna("MISSING"))

channels = T.stream_aggregate(files, ["created_date", "open_data_channel_type"], channel_by_year)
channel_share = channels.div(channels.sum(axis=1), axis=0) * 100
ax = channel_share.plot(kind="bar", stacked=True, figsize=(8, 3.8), colormap="Purples_r", width=0.8)
ax.set(ylabel="% of requests", xlabel="", title="Reporting channel by year")
ax.legend(bbox_to_anchor=(1, 1), frameon=False)
save(ax.figure, "01c_channel_by_year")
channel_share.round(1)

## 5. Seasonality: the motivation for the weather idea
For each complaint type, this shows the share of its requests that fall in each calendar month. It uses full years only (2020–2025) so the current partial year doesn't bias the picture.

In [ ]:
types_of_interest = list(dict.fromkeys(list(prof["complaint_type"].head(12).index) + config.COMPLAINT_TYPES))

def type_by_month(df):
    d = df[df["complaint_type"].isin(types_of_interest) & (df["created_date"] < "2026")]
    return pd.crosstab(d["complaint_type"], d["created_date"].str[5:7].astype(int))

season = T.stream_aggregate(files, ["created_date", "complaint_type"], type_by_month)
season_share = season.div(season.sum(axis=1), axis=0) * 100
season_share = season_share.loc[season_share[[12, 1, 2]].sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 6.5))
sns.heatmap(season_share, cmap="Purples", annot=True, fmt=".0f", cbar_kws={"label": "% of the type's requests"}, ax=ax)
ax.set(xlabel="Calendar month", ylabel="", title="When does each complaint happen? (2020–2025)")
save(fig, "01d_seasonality_heatmap")

**Reading it:**
- **HEAT/HOT WATER** is concentrated in Dec–Feb.
- **Street noise, rodents, water system and dirty conditions** peak in summer.
- **Residential noise** is spread fairly evenly across the year.

Strong seasonality suggests weather may drive these types. Season alone can't separate weather from holidays, school calendars and so on, though. That's why the analysis later compares a cold day with a *normal day in the same month*.

## 6. Closure time and data-quality problems

In [ ]:
def quality_by_year(df):
    created = pd.to_datetime(df["created_date"], errors="coerce", format="ISO8601")
    closed = pd.to_datetime(df["closed_date"], errors="coerce", format="ISO8601")
    hours = (closed - created).dt.total_seconds() / 3600
    flags = pd.DataFrame({
        "rows": 1,
        "no_closed_date": closed.isna(),
        "closed_before_created": hours < 0,
        "closed_in_0_hours": hours == 0,
        "open_over_1_year": hours > 24 * 365,
        "no_coordinates": df["latitude"].isna(),
        "no_bbl": df["bbl"].isna(),
        "borough_unspecified": df["borough"].fillna("").str.upper().eq("UNSPECIFIED"),
    })
    return flags.groupby(df["created_date"].str[:4]).sum()

quality = T.stream_aggregate(files, ["created_date", "closed_date", "latitude", "bbl", "borough"], quality_by_year)
quality_pct = quality.drop(columns="rows").div(quality["rows"], axis=0) * 100
quality_pct.round(2)

In [ ]:
def closure_sample(df):
    created = pd.to_datetime(df["created_date"], errors="coerce", format="ISO8601")
    closed = pd.to_datetime(df["closed_date"], errors="coerce", format="ISO8601")
    out = pd.DataFrame({"agency": df["agency"], "hours": (closed - created).dt.total_seconds() / 3600})
    return out[out["hours"] > 0].sample(frac=0.05, random_state=0)

closures = T.stream_aggregate(files, ["created_date", "closed_date", "agency"], closure_sample, combine="concat")
top_agencies = closures["agency"].value_counts().head(10).index
order = closures[closures.agency.isin(top_agencies)].groupby("agency")["hours"].median().sort_values().index

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=closures[closures.agency.isin(top_agencies)], x="hours", y="agency", order=order,
            showfliers=False, color="#d9c2ea", ax=ax)
ax.set_xscale("log")
ax.set(xlabel="Hours from created to closed (log scale, 5% sample)", ylabel="",
       title="Closure time differs by orders of magnitude between agencies")
save(fig, "01e_closure_by_agency")

## 7. Decisions for the cleaning notebook
Based on this exploration:

| Decision | Why |
|---|---|
| **Complaint types:** HEAT/HOT WATER, Sewer, Water System, Snow or Ice, Noise – Street/Sidewalk, Noise – Residential, Rodent, Dirty Condition | Strongly seasonal, with plausible weather mechanisms. *Residential noise* is kept as a **control** (no expected weather effect). |
| **Years: 2023–2025** | Three complete years, including two complete heating seasons (Oct–May). Reporting channels are more stable than in 2020–21 (COVID period). |
| **About 17 columns** | Dates, type, agency, status, location (ZIP, BBL, coordinates, borough) and channel. The sparse columns are dropped. |
| **Cleaning must handle** | Placeholder values (`Unspecified`), BBL formats, invalid ZIPs and coordinates, and odd closure times (0 hours, negative, over a year). These are *flagged*, not silently deleted. |

The scope lives in `src/config.py`. → Next: **02_clean_311.ipynb**